
#  Ingestão de Dados (Camada Bronze)

Nesta aula, vamos construir a **Camada Bronze** do nosso data lakehouse para a **AluMax** (empresa de atendimento omnichannel).

O objetivo desta etapa é capturar os arquivos brutos de atendimento (que chegam em formato CSV com dados complexos e JSONs) e armazená-los de forma segura e incremental.

---

## Principais Funções

Para quem está começando agora no Databricks e PySpark, destacamos abaixo as quatro funções principais utilizadas no script de ingestão e o que cada uma faz:

### 1. `.format("delta")` (Delta Lake)

* **O que faz:** Salva os dados no formato padrão do Databricks chamado **Delta Lake**.
* **Por que usamos:** Diferente de salvar em um CSV comum ou Parquet simples, o formato Delta traz superpoderes para a nossa Camada Bronze, como transações seguras (ACID) e histórico de versões dos dados.

### 2. `current_timestamp()` (Metadados de Auditoria)

* **O que faz:** Captura a data e a hora exatas em que o registro foi processado pelo pipeline.
* **Por que usamos:** Na camada Bronze, **nunca apagamos o dado original**, mas adicionar essa coluna nos ajuda a auditar exatamente quando a informação entrou no nosso sistema.

### 3. `_metadata.file_path` (Rastreabilidade)

* **O que faz:** Adiciona uma coluna extra informando o nome exato do arquivo de origem de cada linha de dado.
* **Por que usamos:** Se houver algum erro de dado corrompido no futuro, conseguiremos rastrear exatamente qual arquivo gerou o problema.

## Promp para o Genie do Databricks
Atue como um Engenheiro de Dados Sênior e Especialista em Databricks. 
Preciso que você escreva um script Python utilizando **Spark** (em um notebook Databricks) para realizar a **Ingestão da Camada Bronze** de um pipeline de dados.

### Contexto do Projeto:
- **Empresa:** AluMax (atendimento omnichannel: WhatsApp, chat, telefone e e-mail).
- **Dados de Entrada:** Um arquivo .csv localizado no Unity Catalog Volume em: `/Volumes/workspace/raw/bronze/input/atendimentos_alumax.csv`
- **Colunas esperadas:** id_interacao, cliente_id, canal, departamento, status, data_hora e payload_detalhes (JSON).

### Requisitos Técnicos Obrigatórios:
1. **Criação do Schema/Database:** Crie o schema `raw` dentro do catálogo `workspace` (caso ele não exista) utilizando o comando `CREATE DATABASE IF NOT EXISTS workspace.raw;`.
2. **Criação da Tabela Bronze (Delta Lake):** Utilize comandos Spark SQL para ler o arquivo CSV diretamente do Volume e criar/inserir na tabela gerenciada `workspace.raw.bronze_atendimentos_alumax`.
3. **Metadados de Auditoria:** Adicione colunas na tabela utilizando funções nativas do SQL para registrar o momento da ingestão (`current_timestamp()`) e utilize a função `._metadata.file_path` para adicionar o nome do arquivo origem


In [0]:
# ========================================
# PASSO 1: Criar Schema/Database
# ========================================

spark.sql("""
    CREATE DATABASE IF NOT EXISTS workspace.raw
    COMMENT 'Schema para dados brutos (Bronze Layer) do projeto AluMax'
""")

print("✅ Schema 'workspace.raw' criado/verificado com sucesso!")

In [0]:
# ========================================
# PASSO 2: Ingestão do CSV com Metadados de Auditoria
# ========================================

# Caminho do arquivo CSV no Unity Catalog Volume
csv_path = "/Volumes/workspace/raw/bronze/input/atendimentos_alumax.csv"

# Leitura do CSV com inferência de schema
df_source = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .load(csv_path)

# Adicionar colunas de auditoria
from pyspark.sql.functions import current_timestamp, col

df_bronze = df_source \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

# Exibir preview dos dados
print(f"📊 Total de registros lidos: {df_bronze.count()}")
print("\n🔍 Preview dos dados com metadados de auditoria:\n")
df_bronze.show(5, truncate=False)

In [0]:
df_bronze.display()

In [0]:
# ========================================
# PASSO 3: Criar/Inserir na Tabela Bronze (Delta Lake)
# ========================================

# Nome da tabela gerenciada
table_name = "workspace.raw.bronze_atendimentos_alumax"

# Escrever no Delta Lake (modo overwrite para primeira execução)
# Para ingestões incrementais futuras, considere usar modo 'append'
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Tabela '{table_name}' criada/atualizada com sucesso!")
print(f"📁 Formato: Delta Lake (ACID compliant)")
print(f"📅 Metadados de auditoria incluídos: _ingestion_timestamp, _source_file")

In [0]:
# ========================================
# PASSO 4: Validação e Visualização da Tabela Bronze
# ========================================

# Verificar o schema da tabela criada
print("📊 Schema da Tabela Bronze:\n")
spark.table("workspace.raw.bronze_atendimentos_alumax").printSchema()

# Contar registros
total_records = spark.table("workspace.raw.bronze_atendimentos_alumax").count()
print(f"\n✅ Total de registros na tabela: {total_records}")

# Visualizar amostra dos dados
print("\n🔍 Amostra dos primeiros 5 registros:\n")
df_sample = spark.table("workspace.raw.bronze_atendimentos_alumax").limit(5)
display(df_sample)